In [ ]:
# uv add weaviate-client==4.18.3
# docker-compose up -d   启动服务
# docker-compose down    关闭服务

In [2]:
import weaviate
from weaviate.classes.config import Configure,Property,DataType

In [1]:
# 准备数据
items = [
    {"type": "phone", "id": "用户A", "number": 13800001234},
    {"type": "phone", "id": "用户B", "number": 13800005678},
    {"type": "order", "id": "订单1001", "number": 203011010001},
    {"type": "order", "id": "订单1002", "number": 203011010123},
    {"type": "order", "id": "订单2001", "number": 203012150045},
    {"type": "phone", "id": "用户C", "number": 13912345678},
    {"type": "phone", "id": "用户D", "number": 13798765432},
    {"type": "order", "id": "订单3001", "number": 205001020333},
    {"type": "order", "id": "订单3002", "number": 205001020777},
    {"type": "phone", "id": "用户E", "number": 13622223333},
]

In [8]:
def create_number_collection(client:weaviate.WeaviateClient):
    '''
    创建集合
    '''
    client.collections.delete('NumberItem')
    numbers = client.collections.create(
        name = 'NumberItem',
        vectorizer_config= Configure.Vectorizer.none(),   # 不使用内置模型
        properties=[
            Property(name='type',data_type=DataType.TEXT),
            Property(name='item_id',data_type=DataType.TEXT),
            Property(name='number',data_type=DataType.NUMBER)
        ]
    )
    return numbers

In [10]:
def insert_data(numers_collection):
    '''插入数据'''
    with numers_collection.batch.fixed_size(batch_size=200) as batch:
        for item in items:
            batch.add_object(
                vector = [float(item['number'])],
                properties={"type": item["type"],
                            "item_id": item["id"],
                            "number": item["number"] 
                            })
          

In [17]:
def query_near_vector_all(numers_collection,query_number):
    '''查询数据'''
    response = numers_collection.query.near_vector(
        near_vector = [float(query_number)],
        limit=5
    )
    print(f"查询数字: {query_number}")
    print("Top-5 最相近的记录（全部类型）：")
    print(response)
    for i,item in enumerate(response.objects,start=1):
        props = item.properties
        print(
            f"第{i}条：类型={props['type']}，编号={props['item_id']}，数字={props['number']}"
        )

In [23]:
from weaviate.classes.query import Filter

def query_near_vector_filtered(numbers_collection,query_number):
    '''查询数据, 向量检索+payload过滤'''
    where_filter = Filter.by_property('type').equal('order')   # 过滤条件

    response = numbers_collection.query.near_vector(
            near_vector = [float(query_number)],
            limit=5,
            filters=where_filter
        )
    print(f"查询数字: {query_number}")
    print("Top-5 最相近的记录（全部类型） + payload过滤：")
    print(response)
    for i,item in enumerate(response.objects,start=1):
        props = item.properties
        print(
            f"第{i}条：类型={props['type']}，编号={props['item_id']}，数字={props['number']}"
        )

In [34]:

def query_near_vector_graphql(client,query_number):
    '''查询数据, 向量检索+payload过滤'''
    graphql_query = f"""
        {{
        Get {{
            NumberItem(
            nearVector: {{
                vector: [{float(query_number)}]
            }}
            where: {{
                path: ["type"]
                operator: Equal
                valueString: "order"
            }}
            limit: 5
            ) {{
            type
            item_id
            number
            }}
        }}
        }}
    """
    response= client.graphql_raw_query(graphql_query)
    for obj in response.get['NumberItem']:
        print(f"{obj['item_id']} {obj['number']} {obj['type']}")

In [35]:
def main():
    with weaviate.connect_to_local() as client:
        # 创建集合
        numbers = create_number_collection(client)
        # 插入数据
        insert_data(numbers)

        # 查询数据
        query_number = 205001020500
        # query_near_vector_all(numbers,query_number)

        # query_near_vector_filtered(numbers,query_number)

        query_near_vector_graphql(client,query_number)

main()

订单3001 205001020333 order
订单3002 205001020777 order
订单1002 203011010123 order
订单2001 203012150045 order
订单1001 203011010001 order
